# Análisis de Comportamiento por Mes de Registro

Cargamos el dataset limpio y preparamos la columna `reg_month` para agrupar usuarios según el mes en que se registraron.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('clean_product_activity.csv')
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df['reg_month'] = df['created_at'].dt.month_name()

month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

print(f'Total filas: {len(df)} | Usuarios unicos: {df["user_id"].nunique()}')
df.head()

Total filas: 8509 | Usuarios unicos: 1996


,user_id,created_at,country,plan_type,user_age,post_id,post_category,post_created_at,votes_received,user_total_posts,days_since_signup,device_type,days_since_signup_calc,month,reg_month
0,U01988,2025-02-18 02:07:44,PY,pro,26.0,P0008515,sports,2025-05-07 20:55:28,7,16,78,mobile,78.0,2025-05,February
1,U00236,2025-06-22 07:49:10,BR,free,27.0,P0001023,tech,2025-09-13 20:31:06,1,9,83,web,83.0,2025-09,June
2,U00791,2024-02-12 02:45:45,CL,free,28.0,P0003405,tech,2024-02-14 05:17:48,11,2,2,mobile,2.0,2024-02,February
3,U01522,2024-09-22 07:06:50,US,free,16.0,P0006524,finance,2024-09-24 07:51:34,5,2,2,web,2.0,2024-09,September
4,U01092,2025-07-18 02:27:52,PY,free,NaN,P0004665,education,2025-07-24 04:56:56,7,2,6,mobile,6.0,2025-07,July


## 1. Usuarios registrados por mes
Contamos la cantidad de usuarios únicos que se registraron en cada mes del año, sin importar el año, para detectar estacionalidad en los registros.

In [2]:
users_month = df.drop_duplicates('user_id').groupby('reg_month')['user_id'].count().reindex(month_order)
display(users_month.to_frame('usuarios'))

,usuarios
reg_month,
January,186
February,165
March,192
April,169
May,170
June,149
July,165
August,174
September,147


## 2. Promedio de posts y votos por mes de registro
Calculamos el promedio de `user_total_posts` y `votes_received` agrupados por mes de registro para ver si los usuarios de ciertos meses son más activos o reciben más interacción.

In [3]:
metrics = df.groupby('reg_month').agg(
    avg_posts=('user_total_posts', 'mean'),
    avg_votes=('votes_received', 'mean')
).reindex(month_order).round(2)
display(metrics)

,avg_posts,avg_votes
reg_month,,
January,10.35,6.90
February,8.02,7.05
March,8.67,6.60
April,8.95,6.82
May,7.59,6.97
June,8.64,6.85
July,8.82,7.07
August,7.41,6.96
September,8.26,6.98


## 3. Dias activos promedio (retencion)
Usamos `days_since_signup_calc` (calculado a partir de las fechas reales) en lugar de `days_since_signup` que contiene datos no confiables. Esto muestra cuántos días en promedio pasan entre el registro y la publicación de un post.

In [4]:
retention = df.groupby('reg_month')['days_since_signup_calc'].mean().reindex(month_order).round(2)
display(retention.to_frame('avg_dias'))

,avg_dias
reg_month,
January,39.56
February,31.66
March,31.80
April,28.07
May,26.04
June,28.46
July,30.85
August,24.79
September,27.66


## 4. Tipo de plan por mes de registro
Analizamos la proporción de cada tipo de plan (free, pro, enterprise) según el mes de registro para identificar si en ciertos meses se atraen más usuarios de pago.

In [5]:
plan_ct = pd.crosstab(df['reg_month'], df['plan_type'], normalize='index').reindex(month_order) * 100
display(plan_ct.round(1))

plan_type,enterprise,free,pro
reg_month,,,
January,4.5,72.7,22.8
February,2.7,80.9,16.3
March,5.6,75.1,19.3
April,2.0,80.5,17.5
May,5.3,77.9,16.8
June,2.5,71.5,26.0
July,2.2,78.3,19.6
August,4.8,74.3,20.9
September,1.5,82.8,15.7


## 5. Resumen general
Tabla consolidada con las métricas clave por mes de registro: cantidad de usuarios únicos, promedio de posts, votos y días activos calculados.

In [6]:
resumen = df.groupby('reg_month').agg(
    usuarios=('user_id', 'nunique'),
    avg_posts=('user_total_posts', 'mean'),
    avg_votos=('votes_received', 'mean'),
    avg_dias_activo=('days_since_signup_calc', 'mean')
).reindex(month_order).round(2)

resumen

,usuarios,avg_posts,avg_votos,avg_dias_activo
reg_month,,,,
January,186,10.35,6.90,39.56
February,165,8.02,7.05,31.66
March,192,8.67,6.60,31.80
April,169,8.95,6.82,28.07
May,170,7.59,6.97,26.04
June,149,8.64,6.85,28.46
July,165,8.82,7.07,30.85
August,174,7.41,6.96,24.79
September,147,8.26,6.98,27.66
